# 分布式训练

> 上一节算出 7B 模型全量训练大约需要 112 GB 显存（参数 + 梯度 + AdamW 状态），单张 A100 80 GB 装不下，更不用说还要留出激活值的空间。算力上去了，单卡的物理上限成了瓶颈。
>
> 这一节回答多卡如何分工。从最朴素的数据并行（DDP）出发，看清楚每张卡上哪些显存是冗余副本；再用 ZeRO 的三个 Stage 把冗余逐层切到多卡上；最后对比 FSDP 和 DeepSpeed 这两种工程实现，以及 Accelerate 在上层做的统一封装。

把训练搬到多张 GPU 上有三种切法。数据并行（Data Parallel）每张卡持有完整模型副本，只把数据切分；模型并行（Model Parallel）把模型本身按维度切开，每张卡只负责一部分计算；流水并行（Pipeline Parallel）按层把模型分段，不同段在不同卡上接力执行。三种范式各有适用场景，大型预训练通常混合使用。

本节聚焦数据并行的演进路线。原因有两个：第一，它是最常用、也最容易理解的并行方式；第二，ZeRO、FSDP、DeepSpeed 这些工业界主流方案，本质都是在数据并行的基础上把显存切得更细。理解了 DDP 的冗余在哪，ZeRO 三个 Stage 在做什么，剩下两种实现的差异就一目了然。


## 1. 单卡的极限：再算一遍显存账单

先把显存账单摆在桌面上。AdamW + 混合精度训练下，每个参数大约占用 16 bytes 显存：FP16 参数 2 bytes、FP16 梯度 2 bytes、FP32 master 权重 4 bytes、AdamW 一阶矩 m 4 bytes、AdamW 二阶矩 v 4 bytes。这部分是「与训练 step 数无关」的固定开销，加上激活值才是完整显存。


In [1]:
# === 单卡显存账单：7B 模型 ===
P = 7e9  # 7B 参数

param_fp16   = 2 * P   # FP16 参数
grad_fp16    = 2 * P   # FP16 梯度
master_fp32  = 4 * P   # FP32 master 权重
adam_m       = 4 * P   # AdamW 一阶矩
adam_v       = 4 * P   # AdamW 二阶矩

fixed_bytes = param_fp16 + grad_fp16 + master_fp32 + adam_m + adam_v
fixed_gb = fixed_bytes / 1e9

print(f"模型参数 (FP16):  {param_fp16 / 1e9:.1f} GB")
print(f"梯度 (FP16):      {grad_fp16 / 1e9:.1f} GB")
print(f"master 权重 (FP32): {master_fp32 / 1e9:.1f} GB")
print(f"AdamW m (FP32):   {adam_m / 1e9:.1f} GB")
print(f"AdamW v (FP32):   {adam_v / 1e9:.1f} GB")
print(f"固定开销小计:     {fixed_gb:.0f} GB (= 16 × P bytes)")
print()
print(f"单张 A100 (80 GB) 装不下 {fixed_gb:.0f} GB 的固定开销，还要给激活值留空间。")


模型参数 (FP16):  14.0 GB
梯度 (FP16):      14.0 GB
master 权重 (FP32): 28.0 GB
AdamW m (FP32):   28.0 GB
AdamW v (FP32):   28.0 GB
固定开销小计:     112 GB (= 16 × P bytes)

单张 A100 (80 GB) 装不下 112 GB 的固定开销，还要给激活值留空间。


## 2. 数据并行（DDP）

最直接的思路是数据并行（Distributed Data Parallel，DDP）：每张卡持有一份完整模型副本，把数据集切成 N 份分给 N 张卡，各自独立做前向反向，反向结束后用 all-reduce 把梯度平均一下。每张卡看到不同的 batch，但更新同一份模型——本质上等价于用 N 倍 batch size 训练。

吞吐量线性增长是 DDP 的最大优点：8 张卡的吞吐接近单卡的 8 倍。代价是显存没有节省，每张卡都存着完整的 112 GB 固定开销。8 张卡的总显存占用是 8 × 112 = 896 GB，其中 7/8 是冗余——同样的优化器状态、同样的 master 权重，在每张卡上都存了一份。


In [2]:
# === DDP 的显存代价 ===
N = 8  # 8 张卡
P = 7e9

per_card_bytes = 16 * P   # DDP 下每卡都是完整副本
total_bytes = N * per_card_bytes

print(f"DDP 配置: {N} 张卡，每张卡 = {per_card_bytes / 1e9:.0f} GB 固定开销")
print(f"总显存占用 = {N} × {per_card_bytes / 1e9:.0f} = {total_bytes / 1e9:.0f} GB")
print(f"其中冗余部分 = {total_bytes / 1e9 - per_card_bytes / 1e9:.0f} GB（{(N-1)/N*100:.0f}% 是副本）")
print()
print("关键观察：DDP 把吞吐做到了线性增长，但显存完全没省。")
print("如果能把这些冗余副本切到多卡上，就能训更大的模型。")


DDP 配置: 8 张卡，每张卡 = 112 GB 固定开销
总显存占用 = 8 × 112 = 896 GB
其中冗余部分 = 784 GB（88% 是副本）

关键观察：DDP 把吞吐做到了线性增长，但显存完全没省。
如果能把这些冗余副本切到多卡上，就能训更大的模型。


DDP 的反向结束后，每张卡上的梯度来自不同的 micro-batch，数值不同。要让所有卡更新到同一份模型，必须把梯度同步一次——这就是 all-reduce 做的事：所有卡把自己的梯度发出去，每张卡收到全部梯度后求平均，最终每张卡上的梯度完全一致。

下面用两个「虚拟 rank」模拟这个过程：各自算完反向得到不同梯度，all-reduce 后两者相同。这不是真正的多进程 DDP，但它把 all-reduce 的核心操作——求平均——直接摊在桌上。

In [1]:
# === DDP all-reduce 可观察 demo：两张卡，梯度不同 → 平均后一致 ===
import torch

torch.manual_seed(42)

# 模拟一个简单模型的两份梯度（来自不同 micro-batch）
d_model = 8
grad_rank0 = torch.randn(d_model) * 2.0 + 1.0    # rank 0 的反向结果
grad_rank1 = torch.randn(d_model) * 2.0 - 1.0    # rank 1 的反向结果

print("=== all-reduce 前 ===")
print(f"rank 0 梯度: {grad_rank0}")
print(f"rank 1 梯度: {grad_rank1}")
print(f"两 rank 梯度是否相同？ {torch.allclose(grad_rank0, grad_rank1)}")
print()

# all-reduce 的核心操作：所有 rank 的梯度求平均
grad_avg = (grad_rank0 + grad_rank1) / 2.0

# all-reduce 后每张卡都拿到这份平均值
grad_rank0_after = grad_avg.clone()
grad_rank1_after = grad_avg.clone()

print("=== all-reduce 后（每卡都拿到平均梯度）===")
print(f"rank 0 梯度: {grad_rank0_after}")
print(f"rank 1 梯度: {grad_rank1_after}")
print(f"两 rank 梯度是否相同？ {torch.allclose(grad_rank0_after, grad_rank1_after)}")
print()
# 验证：平均后每张卡的梯度 = 各卡原始梯度的算术平均
print(f"手动验证 rank0 梯度 == 平均？ {torch.allclose(grad_rank0_after, grad_avg)}")
print()
print("关键观察：all-reduce 把 N 张卡的不同梯度变成了同一份平均值。")
print("每张卡用这份平均梯度更新参数，所有卡上的模型保持完全一致。")
print("这就是 DDP 在反向之后、optimizer.step() 之前做的事。")

=== all-reduce 前 ===
rank 0 梯度: tensor([ 1.6734,  1.2576,  1.4689,  1.4607, -1.2457,  0.6273,  5.4164, -0.2760])
rank 1 梯度: tensor([-0.0767, -0.4653,  0.0698,  0.6187,  1.2206, -4.3796, -2.9779,  0.9159])
两 rank 梯度是否相同？ False

=== all-reduce 后（每卡都拿到平均梯度）===
rank 0 梯度: tensor([ 0.7983,  0.3962,  0.7694,  1.0397, -0.0126, -1.8761,  1.2192,  0.3200])
rank 1 梯度: tensor([ 0.7983,  0.3962,  0.7694,  1.0397, -0.0126, -1.8761,  1.2192,  0.3200])
两 rank 梯度是否相同？ True

手动验证 rank0 梯度 == 平均？ True

关键观察：all-reduce 把 N 张卡的不同梯度变成了同一份平均值。
每张卡用这份平均梯度更新参数，所有卡上的模型保持完全一致。
这就是 DDP 在反向之后、optimizer.step() 之前做的事。


## 3. ZeRO：把冗余切到多卡

ZeRO（Zero Redundancy Optimizer）是微软 2020 年提出的方法。它的出发点很简单：DDP 下每张卡上的固定开销 16P bytes 由三部分组成——优化器状态（12P bytes，含 master 权重、m、v）、梯度（2P bytes）、参数（2P bytes）。这三部分都是冗余的，可以分别切到多卡上。

ZeRO 把切分过程拆成三个递进的 Stage，每升一级多切一类。Stage 1 切优化器状态，Stage 2 再切梯度，Stage 3 再切参数。下面手算每个 Stage 在 8 卡下能省多少显存。


### 3.1 Stage 1：切优化器状态

AdamW 的优化器状态（master 权重、m、v）共 12P bytes。Stage 1 把这 12P 切到 N 卡，每卡只存 12P/N。剩下的梯度（2P）和参数（2P）仍然每卡完整保留。

切完之后，每卡显存 = 2P（参数）+ 2P（梯度）+ 12P/N（优化器状态）。


In [3]:
# === ZeRO Stage 1 显存手算 ===
P = 7e9
N = 8

s1_optimizer = 12 * P / N
s1_param = 2 * P
s1_grad = 2 * P
s1_total = s1_optimizer + s1_param + s1_grad

print(f"Stage 1 单卡显存（7B × {N} 卡）：")
print(f"  参数 (FP16，完整):     {s1_param / 1e9:.1f} GB")
print(f"  梯度 (FP16，完整):     {s1_grad / 1e9:.1f} GB")
print(f"  优化器状态 (1/{N} 切片): {s1_optimizer / 1e9:.1f} GB")
print(f"  单卡小计:              {s1_total / 1e9:.1f} GB")
print()
print(f"相比 DDP 的 112 GB，Stage 1 省到 {s1_total / 1e9:.1f} GB。")
print("代价：反向结束后要 reduce-scatter 梯度，让每卡只保留自己那一片优化器对应的梯度。")


Stage 1 单卡显存（7B × 8 卡）：
  参数 (FP16，完整):     14.0 GB
  梯度 (FP16，完整):     14.0 GB
  优化器状态 (1/8 切片): 10.5 GB
  单卡小计:              38.5 GB

相比 DDP 的 112 GB，Stage 1 省到 38.5 GB。
代价：反向结束后要 reduce-scatter 梯度，让每卡只保留自己那一片优化器对应的梯度。


### 3.2 Stage 2：再切梯度

Stage 1 留下了一个对称性破缺：梯度虽然只有 2P bytes，但每张卡都保留了完整副本。Stage 2 把梯度也按 N 卡切片，每卡只保留自己优化器那一片对应的梯度。

切完之后，每卡显存 = 2P（参数）+ 2P/N（梯度）+ 12P/N（优化器状态）。


In [4]:
# === ZeRO Stage 2 显存手算 ===
P = 7e9
N = 8

s2_optimizer = 12 * P / N
s2_grad = 2 * P / N
s2_param = 2 * P
s2_total = s2_optimizer + s2_grad + s2_param

print(f"Stage 2 单卡显存（7B × {N} 卡）：")
print(f"  参数 (FP16，完整):     {s2_param / 1e9:.1f} GB")
print(f"  梯度 (1/{N} 切片):      {s2_grad / 1e9:.2f} GB")
print(f"  优化器状态 (1/{N} 切片): {s2_optimizer / 1e9:.1f} GB")
print(f"  单卡小计:              {s2_total / 1e9:.1f} GB")


Stage 2 单卡显存（7B × 8 卡）：
  参数 (FP16，完整):     14.0 GB
  梯度 (1/8 切片):      1.75 GB
  优化器状态 (1/8 切片): 10.5 GB
  单卡小计:              26.2 GB


### 3.3 Stage 3：再切参数

Stage 3 把最后一份冗余——参数本身——也切了。每卡只持有 1/N 的参数。前向和反向用到参数时，临时 all-gather 出完整 layer，用完即丢。

切完之后，每卡显存 = 16P/N（三类全部切片）。这就是 ZeRO 的极限。


In [5]:
# === ZeRO Stage 3 显存手算 ===
P = 7e9
N = 8

s3_total = 16 * P / N

print(f"Stage 3 单卡显存（7B × {N} 卡）：")
print(f"  参数 + 梯度 + 优化器状态 (全部 1/{N} 切片): {s3_total / 1e9:.1f} GB")
print()
print(f"相比 DDP 的 112 GB，Stage 3 省到 {s3_total / 1e9:.1f} GB。")
print("代价：每次前向/反向都要 all-gather 出当前层的完整参数，通信量明显增加。")


Stage 3 单卡显存（7B × 8 卡）：
  参数 + 梯度 + 优化器状态 (全部 1/8 切片): 14.0 GB

相比 DDP 的 112 GB，Stage 3 省到 14.0 GB。
代价：每次前向/反向都要 all-gather 出当前层的完整参数，通信量明显增加。


### 三个 Stage 对比

把三个 Stage 在 8 卡、7B 模型下的单卡显存放在一起：


In [6]:
# === ZeRO 三 Stage 总对比 ===
P = 7e9

print(f"{'方案':<22} {'单卡显存':>12} {'相对 DDP':>12}")
print("-" * 50)
configs = [
    ("DDP",           16 * P),
    ("ZeRO Stage 1",  2 * P + 2 * P + 12 * P / 8),
    ("ZeRO Stage 2",  2 * P + 2 * P / 8 + 12 * P / 8),
    ("ZeRO Stage 3",  16 * P / 8),
]
ddp_bytes = 16 * P
for name, total in configs:
    ratio = total / ddp_bytes
    print(f"{name:<22} {total / 1e9:>10.1f} GB   {ratio * 100:>6.1f}%")
print()
print("从 DDP 到 Stage 3，单卡显存压缩到 1/8。通信代价也随 Stage 递增。")


方案                             单卡显存       相对 DDP
--------------------------------------------------
DDP                         112.0 GB    100.0%
ZeRO Stage 1                 38.5 GB     34.4%
ZeRO Stage 2                 26.2 GB     23.4%
ZeRO Stage 3                 14.0 GB     12.5%

从 DDP 到 Stage 3，单卡显存压缩到 1/8。通信代价也随 Stage 递增。


## 4. FSDP：PyTorch 原生的 ZeRO-3

ZeRO 是论文里的算法，工程上有两套主流实现：FSDP 和 DeepSpeed。先看 FSDP。

FSDP（Fully Sharded Data Parallel）是 PyTorch 1.11 起内置的模块，定位就是 ZeRO Stage 3 的官方实现。它把参数、梯度、优化器状态全部分片到各卡，前向反向时按需 all-gather 出完整参数。和 DDP 的 API 非常接近，迁移成本主要在包装那一行：


In [7]:
# === DDP vs FSDP 的最小代码差异（伪代码，单进程无法运行） ===
# 这段代码只是结构对比，不在 notebook 里实际启动多进程

ddp_code = '''
from torch.nn.parallel import DistributedDataParallel as DDP

model = MyModel().cuda()
model = DDP(model)              # ← 关键差异
optimizer = torch.optim.AdamW(model.parameters())
'''

fsdp_code = '''
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = MyModel().cuda()
model = FSDP(model)             # ← 关键差异
optimizer = torch.optim.AdamW(model.parameters())
'''

print("DDP 版:")
print(ddp_code)
print("FSDP 版:")
print(fsdp_code)
print("差异：把 DDP 换成 FSDP，其余训练循环保持不变。")
print("FSDP 内部自动处理 all-gather / reduce-scatter，对外暴露的接口和 DDP 一致。")


DDP 版:

from torch.nn.parallel import DistributedDataParallel as DDP

model = MyModel().cuda()
model = DDP(model)              # ← 关键差异
optimizer = torch.optim.AdamW(model.parameters())

FSDP 版:

from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = MyModel().cuda()
model = FSDP(model)             # ← 关键差异
optimizer = torch.optim.AdamW(model.parameters())

差异：把 DDP 换成 FSDP，其余训练循环保持不变。
FSDP 内部自动处理 all-gather / reduce-scatter，对外暴露的接口和 DDP 一致。


FSDP 的优势是和 PyTorch 主线绑定，跟 NCCL、CUDA 新特性的适配最快。劣势是配置项较多，MixedPrecision、ShardingStrategy、CPUOffload 这些都要自己指定。需要精细控制时是首选；想快速跑起来时，DeepSpeed 的配置驱动方式更省事。

PyTorch 2.x 引入了 FSDP2（`torch.distributed.fsdp.v2`），核心改进是把参数分片从「按 module 树」改为「按 tensor 维度」——不再受 `nn.Module` 嵌套结构的约束，对 MoE 等非均匀结构的支持更好。API 与 FSDP1 不兼容，新项目建议从 FSDP2 起步。

## 5. DeepSpeed：配置驱动的 ZeRO

DeepSpeed 是微软开源的训练框架，把 ZeRO 的 Stage 选择、offload 策略、通信优化都做成 JSON 配置项。训练代码几乎不用动，启动时通过 `--deepspeed_config` 传入即可。

下面是一个启用 ZeRO Stage 2 的最小配置：


In [8]:
# === DeepSpeed 最小配置示例 ===
import json

deepspeed_config = {
    "train_micro_batch_size_per_gpu": 4,
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "none"          # 不 offload 到 CPU，保持 GPU 速度
        }
    },
    "fp16": {
        "enabled": True
    }
}

print("deepspeed_config.json:")
print(json.dumps(deepspeed_config, indent=2, ensure_ascii=False))
print()
print("启动命令：")
print("  deepspeed --num_gpus=8 train.py --deepspeed_config deepspeed_config.json")
print()
print("把 stage 改成 3 就启用了 ZeRO-3（等价于 FSDP）。")
print('offload_optimizer.device 改成 "cpu" 就把优化器状态卸到内存，省更多显存但慢。')


deepspeed_config.json:
{
  "train_micro_batch_size_per_gpu": 4,
  "zero_optimization": {
    "stage": 2,
    "offload_optimizer": {
      "device": "none"
    }
  },
  "fp16": {
    "enabled": true
  }
}

启动命令：
  deepspeed --num_gpus=8 train.py --deepspeed_config deepspeed_config.json

把 stage 改成 3 就启用了 ZeRO-3（等价于 FSDP）。
offload_optimizer.device 改成 "cpu" 就把优化器状态卸到内存，省更多显存但慢。


DeepSpeed 相对 FSDP 多了两类能力：一是 ZeRO-Offload，可以把优化器状态甚至参数卸到 CPU 内存或 NVMe 硬盘，换取更大的有效显存；二是 ZeRO-Infinity，把 offload 推到极致，能在 32 张 GPU 上训万亿参数模型。代价是 CPU↔GPU 的数据搬运开销，速度比纯 GPU 的 FSDP 慢。模型极大、显存实在不够时才用。


## 6. Accelerate：统一封装

写训练脚本时会发现 DDP、FSDP、DeepSpeed 的启动方式都不一样：DDP 用 `torchrun`，FSDP 也用 `torchrun` 但要改包装，DeepSpeed 用 `deepspeed` launcher。换后端就要改代码很烦。

Accelerate 是 HuggingFace 提供的薄层封装，把这些差异藏在一个 `Accelerator` 对象后面。同一份训练代码，配合不同的 `accelerate config` 就能切换 DDP / FSDP / DeepSpeed：


In [9]:
# === Accelerate 的统一训练循环（伪代码） ===
accelerate_code = '''
from accelerate import Accelerator

accelerator = Accelerator()

model, optimizer, dataloader, scheduler = accelerator.prepare(
    model, optimizer, dataloader, scheduler
)

for batch in dataloader:
    loss = model(batch)
    accelerator.backward(loss)        # ← 取代 loss.backward()
    optimizer.step()
    optimizer.zero_grad()
'''

print("训练代码：")
print(accelerate_code)
print()
print("切换后端只需改 accelerate config，代码不动：")
print("  accelerate config  → 交互式选择 DDP / FSDP / DeepSpeed")
print("  accelerate launch train.py  → 用配置好的后端启动")


训练代码：

from accelerate import Accelerator

accelerator = Accelerator()

model, optimizer, dataloader, scheduler = accelerator.prepare(
    model, optimizer, dataloader, scheduler
)

for batch in dataloader:
    loss = model(batch)
    accelerator.backward(loss)        # ← 取代 loss.backward()
    optimizer.step()
    optimizer.zero_grad()


切换后端只需改 accelerate config，代码不动：
  accelerate config  → 交互式选择 DDP / FSDP / DeepSpeed
  accelerate launch train.py  → 用配置好的后端启动


Accelerate 不是另一套分布式算法，它只是个调度层——DDP 模式下底层就是 torch.distributed，FSDP 模式下底层就是 PyTorch FSDP，DeepSpeed 模式下底层就是 DeepSpeed。它的价值是「同一份代码多套后端」，适合做实验对比、或者写跨框架开源项目。


## 7. 三层关系总结

把三套方案放在一起看，它们处在不同的抽象层级：


In [10]:
# === 三层方案对比表（用 print 输出避免 markdown 表格在 PDF 里的渲染问题） ===
rows = [
    ("方案", "抽象层级", "切分粒度", "典型场景"),
    ("PyTorch DDP", "最底层", "无切分（数据并行）", "模型放得下单卡"),
    ("PyTorch FSDP", "底层（PyTorch 内置）", "ZeRO-3，按层包装", "需要精细控制通信"),
    ("DeepSpeed", "中层（配置驱动）", "ZeRO 1/2/3 + offload", "训练超大模型"),
    ("Accelerate", "上层（薄封装）", "不切分，只调度", "一份代码切多种后端"),
]
widths = [16, 22, 26, 30]
for row in rows:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "方案":
        print("-" * len(line))


方案               | 抽象层级                   | 切分粒度                       | 典型场景                          
-------------------------------------------------------------------------------------------------------
PyTorch DDP      | 最底层                    | 无切分（数据并行）                  | 模型放得下单卡                       
PyTorch FSDP     | 底层（PyTorch 内置）         | ZeRO-3，按层包装                | 需要精细控制通信                      
DeepSpeed        | 中层（配置驱动）               | ZeRO 1/2/3 + offload       | 训练超大模型                        
Accelerate       | 上层（薄封装）                | 不切分，只调度                    | 一份代码切多种后端                     


从下往上抽象层级提升，灵活性下降，便利性上升。DDP / FSDP / DeepSpeed 各自完成一类显存切分策略，Accelerate 在它们之上做调度。真实项目里经常混用——比如用 Accelerate 写主脚本，配合 FSDP 或 DeepSpeed 作为后端。


## 8. 并行策略选型：决策树

前面把数据并行（DDP → ZeRO → FSDP/DeepSpeed）的演进线走了一遍。Tensor Parallelism、Pipeline Parallelism、Expert Parallelism 的细节在附录 E 展开。这一节回答一个实操问题：拿到一个模型，怎么选并行方案？

决策可以归纳为四步，核心变量是模型规模、是否 MoE、节点间带宽。

**第一步：模型放得下单卡吗？**

先算固定显存账单（第 1 节的方法）。如果 16P bytes + 激活值 < 单卡显存，直接用 DDP。吞吐线性增长，代码改动最少。

**第二步：需要切优化器状态吗？**

单卡放不下时，先开 ZeRO Stage 2（FSDP 或 DeepSpeed）。ZeRO-2 把优化器状态和梯度切到多卡，通信代价等于一次 all-reduce，额外开销小。7B-13B 模型的全量微调在 8 卡节点内用 ZeRO-2 即可。

**第三步：参数也要切？**

ZeRO-3（FSDP 或 DeepSpeed ZeRO-3）把参数也切片。通信量仍是等效一次 all-reduce，但每次前向需要 all-gather 参数，对带宽更敏感。节点间是 InfiniBand（≥200 GB/s）时跨节点 ZeRO-3 可行；如果是以太网（≤100 GB/s），改为 3D 并行——节点内 TP、节点间 PP/DP。

**第四步：模型是 MoE 吗？**

如果是 MoE（Mixtral、DeepSeek-V3），在第三步基础上叠加 Expert Parallelism（EP）。EP 把不同 expert 分到不同卡，通信是 all-to-all（附录 E 第 5 节）。EP 并行度通常取 expert 数量的约数：256 expert 配 EP=64，每卡管 4 个 expert。

下面用代码把这四步做成一个查询表——给定模型规模和类型，输出推荐方案。

In [1]:
# === 并行策略选型参考表 ===
# 场景 → 推荐方案，核心判据：模型规模 + MoE + 节点间带宽

print(f"{'模型规模':<16}{'MoE?':<8}{'单节点够?':<10}{'推荐方案':<36}")
print("-" * 70)

scenarios = [
    ("< 7B",      "否", "是", "DDP（单卡或多卡，不用 ZeRO）"),
    ("7B ~ 13B",  "否", "是", "ZeRO-2（FSDP 或 DeepSpeed Stage 2）"),
    ("13B ~ 70B", "否", "是", "ZeRO-3 / FSDP（节点内 NVLink）"),
    ("70B ~ 200B","否", "否", "TP=8(节点内) + PP(节点间) + DP"),
    ("200B+",     "否", "否", "TP + PP + DP 三维混合，TP 锁节点内"),
    ("7B ~ 70B",  "是", "是", "ZeRO-2 + EP（EP 度 = expert 数 / 每卡 expert 数）"),
    ("70B ~ 400B","是", "否", "EP + TP(节点内) + PP + DP，EP 先确定"),
    ("400B+",     "是", "否", "EP + TP + PP + DP + SP，DualPipe 压 bubble"),
]

for model, moe, fit_node, strategy in scenarios:
    print(f"{model:<16}{moe:<8}{fit_node:<10}{strategy:<36}")

print()
print("核心原则：")
print("  1. TP 只放在节点内（all-reduce 对延迟敏感，NVLink 内才划算）")
print("  2. EP 先确定（expert 数决定 EP 上限，然后剩余维度分给 TP/PP/DP）")
print("  3. PP 的 M（micro-batch 数）取 4-8 倍 stage 数，压 bubble 到可忽略")
print("  4. DP 填充剩余 GPU，受 critical batch size 约束不能无限加")

模型规模            MoE?    单节点够?     推荐方案                                
----------------------------------------------------------------------
< 7B            否       是         DDP（单卡或多卡，不用 ZeRO）                  
7B ~ 13B        否       是         ZeRO-2（FSDP 或 DeepSpeed Stage 2）    
13B ~ 70B       否       是         ZeRO-3 / FSDP（节点内 NVLink）           
70B ~ 200B      否       否         TP=8(节点内) + PP(节点间) + DP            
200B+           否       否         TP + PP + DP 三维混合，TP 锁节点内           
7B ~ 70B        是       是         ZeRO-2 + EP（EP 度 = expert 数 / 每卡 expert 数）
70B ~ 400B      是       否         EP + TP(节点内) + PP + DP，EP 先确定       
400B+           是       否         EP + TP + PP + DP + SP，DualPipe 压 bubble

核心原则：
  1. TP 只放在节点内（all-reduce 对延迟敏感，NVLink 内才划算）
  2. EP 先确定（expert 数决定 EP 上限，然后剩余维度分给 TP/PP/DP）
  3. PP 的 M（micro-batch 数）取 4-8 倍 stage 数，压 bubble 到可忽略
  4. DP 填充剩余 GPU，受 critical batch size 约束不能无限加


- [ ] DDP 每张卡持有完整模型副本，吞吐线性增长但显存无节省
- [ ] DDP 反向结束后 all-reduce 把 N 张卡的不同梯度平均成同一份，所有卡同步更新
- [ ] ZeRO 把冗余切到多卡：Stage 1 切优化器状态，Stage 2 再切梯度，Stage 3 再切参数
- [ ] 7B 模型在 8 卡下，DDP 单卡 112 GB → ZeRO-3 单卡 14 GB
- [ ] Stage 越高显存越省，但通信代价越大
- [ ] FSDP 是 PyTorch 内置的 ZeRO-3，FSDP2 改为按 tensor 维度分片
- [ ] DeepSpeed 用 JSON 配置 Stage，多了 ZeRO-Offload 能卸载到 CPU/NVMe
- [ ] Accelerate 是上层调度，一份代码切换 DDP / FSDP / DeepSpeed
- [ ] 并行策略选型：模型规模 → ZeRO stage → 是否引入 TP/PP/EP（附录 E）

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：算 ZeRO Stage 3 在 4 卡下的单卡显存**

7B 模型，AdamW 训练，4 张卡。ZeRO Stage 3 下每张卡的固定显存是多少 GB？

小提示：Stage 3 把 16P bytes 全部切片到 N 卡，每卡 16P/N。


In [11]:
# 作业 1：ZeRO Stage 3 在 4 卡下的单卡显存
P = 7e9
N = 4

# TODO: 计算 Stage 3 单卡显存
s3_per_card_gb = 16 * P / N / 1e9

assert s3_per_card_gb is not None, "请先计算 Stage 3 单卡显存"
expected = 16 * P / N / 1e9
assert abs(s3_per_card_gb - expected) < 0.1, f"应为 {expected:.1f} GB"
print(f"✅ 作业 1 通过：")
print(f"   Stage 3 + 4 卡 + 7B：单卡 {s3_per_card_gb:.1f} GB")
print(f"   相比单卡 DDP 的 112 GB，压缩到 {s3_per_card_gb / 112 * 100:.1f}%。")


✅ 作业 1 通过：
   Stage 3 + 4 卡 + 7B：单卡 28.0 GB
   相比单卡 DDP 的 112 GB，压缩到 25.0%。


**作业 2：DDP vs ZeRO-3 总显存对比**

7B 模型，8 张卡。DDP 和 ZeRO Stage 3 各自的**总显存**（所有卡加起来）是多少？为什么 ZeRO-3 看似省了很多，但总显存其实只省了优化器状态那一份？

小提示：DDP 每卡完整副本所以总 = N × 16P；ZeRO-3 每卡 16P/N，总 = 16P。


In [12]:
# 作业 2：DDP vs ZeRO-3 总显存
P = 7e9
N = 8

# TODO: 计算 DDP 总显存和 ZeRO-3 总显存（单位 GB）
ddp_total_gb = N * 16 * P / 1e9
zero3_total_gb = 16 * P / 1e9

assert ddp_total_gb is not None and zero3_total_gb is not None, "请先计算两个总显存"
expected_ddp = N * 16 * P / 1e9
expected_zero3 = 16 * P / 1e9
assert abs(ddp_total_gb - expected_ddp) < 1, f"DDP 总显存应为 {expected_ddp:.0f} GB"
assert abs(zero3_total_gb - expected_zero3) < 1, f"ZeRO-3 总显存应为 {expected_zero3:.0f} GB"
print(f"✅ 作业 2 通过：")
print(f"   DDP 总显存（8 卡）:    {ddp_total_gb:.0f} GB")
print(f"   ZeRO-3 总显存（8 卡）: {zero3_total_gb:.0f} GB")
print(f"   省下来的 {ddp_total_gb - zero3_total_gb:.0f} GB 全是冗余副本。")


✅ 作业 2 通过：
   DDP 总显存（8 卡）:    896 GB
   ZeRO-3 总显存（8 卡）: 112 GB
   省下来的 784 GB 全是冗余副本。


**作业 3：写一个启用 ZeRO Stage 2 的 DeepSpeed 配置**

补全下面的 `deepspeed_config` 字典，要求：启用 ZeRO Stage 2，不 offload 到 CPU，开启 FP16。

小提示：参考第 5 节的示例，关键字段是 `zero_optimization.stage`、`zero_optimization.offload_optimizer.device`、`fp16.enabled`。


In [13]:
# 作业 3：写 ZeRO Stage 2 的 DeepSpeed 配置
import json

deepspeed_config = {
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {"device": "none"},
    },
    "fp16": {"enabled": True},
}

# 验证
assert "zero_optimization" in deepspeed_config, "缺少 zero_optimization 字段"
assert deepspeed_config["zero_optimization"]["stage"] == 2, "stage 应为 2"
assert deepspeed_config["zero_optimization"]["offload_optimizer"]["device"] == "none",     "offload_optimizer.device 应为 'none'"
assert deepspeed_config["fp16"]["enabled"] is True, "fp16.enabled 应为 True"

print("✅ 作业 3 通过：")
print("你的 DeepSpeed 配置：")
print(json.dumps(deepspeed_config, indent=2, ensure_ascii=False))
print()
print("这份配置启用了 ZeRO Stage 2，把梯度和优化器状态切到多卡，参数仍完整保留。")


✅ 作业 3 通过：
你的 DeepSpeed 配置：
{
  "zero_optimization": {
    "stage": 2,
    "offload_optimizer": {
      "device": "none"
    }
  },
  "fp16": {
    "enabled": true
  }
}

这份配置启用了 ZeRO Stage 2，把梯度和优化器状态切到多卡，参数仍完整保留。


## 参考资料

- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020
- [PyTorch FSDP 文档](https://pytorch.org/docs/stable/fsdp.html)
- [DeepSpeed 文档](https://www.deepspeed.ai/)
- [HuggingFace Accelerate 文档](https://huggingface.co/docs/accelerate/)
- Zhao et al., [PyTorch FSDP](https://arxiv.org/abs/2304.11277), 2023
